# PI-CAI Domain Adaptation on Kaggle

This notebook trains the Z-SSMNet model natively on Kaggle using Kaggle Datasets.
It is configured for **Domain Adaptation**, meaning you can train on 2 centers (e.g. RUMC and ZGT) while completely holding out the 3rd center (PCNN) for testing later.

### Prerequisites:
1. Ensure you have added your `picai-pre-processed-dataset` to this notebook (via the **Add Data** button on the right).
2. Ensure you have turned on **Internet Access** in the Kaggle notebook settings.
3. Ensure your Accelerator is set to **GPU T4 x2** or **GPU P100**.

In [ ]:
import os
import sys

# 1. Define Paths for Kaggle Environment
# Kaggle mounts datasets read-only at /kaggle/input/
SOURCE_DATA_DIR = "/kaggle/input/datasets/hemishjain09/picai-pre-processed-dataset"

# Kaggle limits /kaggle/working to 20 GB. 
# We route the massive 70GB preprocessing cache to /tmp (which has ~75GB) to prevent crashes!
WORKSPACE_DIR = "/tmp/PI-CAI_Workspace/baseline"

# We strictly keep RESULTS_FOLDER in /kaggle/working so the model checkpoints are saved!
RESULTS_FOLDER = "/kaggle/working/PI-CAI_Results"

# 2. Domain Adaptation Setup
# ------------------------------------------------------------------
TRAIN_CENTERS = "RUMC,ZGT"

# Sanity Check / Testing limits (set to empty strings "" for a full run)
MAX_CASES = "10"     
MAX_EPOCHS = "1"    

# Export variables to environment
os.environ["SOURCE_DATA_DIR"] = SOURCE_DATA_DIR
os.environ["RESULTS_FOLDER"] = RESULTS_FOLDER
os.environ["WORKSPACE_DIR"] = WORKSPACE_DIR
os.environ["TRAIN_CENTERS"] = TRAIN_CENTERS
os.environ["MAX_CASES"] = MAX_CASES
os.environ["MAX_EPOCHS"] = MAX_EPOCHS

print("Environment configured for Kaggle (using /tmp optimization)!")
print(f"Training on centers: {TRAIN_CENTERS}")

### Optional: Resuming from a 12-Hour Timeout
Kaggle restricts continuous GPU runs to 12 hours. If your run times out:
1. Start a New Version of this notebook.
2. Click **Add Data** -> **Your Work** -> Select the Output of your *previous* run.
3. Set `PREVIOUS_OUTPUT_NAME` to the exact name of your previous run (e.g. `notebook-xyz`).
4. Uncomment and run the cell below to restore your checkpoints into the active working directory before running the pipeline.

In [ ]:
# PREVIOUS_OUTPUT_NAME = "your-previous-notebook-output-name"
# PREV_PATH = f"/kaggle/input/{PREVIOUS_OUTPUT_NAME}"

# import os
# if os.path.exists(PREV_PATH):
#     print("Restoring previous checkpoints...")
#     !cp -rn $PREV_PATH/PI-CAI_Workspace /tmp/  # Note: moving to /tmp instead of /kaggle/working
#     !cp -rn $PREV_PATH/PI-CAI_Results /kaggle/working/
#     print("Restore complete! Now run the pipeline cell.")
# else:
#     print("Could not find previous output. Check the name!")

In [ ]:
# 3. Clean workspaces, clone repositories, and pull fixes
!rm -rf /tmp/PI-CAI_Workspace
!rm -rf /kaggle/working/PI-CAI_Results
!mkdir -p /tmp/PI-CAI_Workspace

import os
if not os.path.exists("/tmp/PI-CAI_Workspace/baseline"):
    !cd /tmp/PI-CAI_Workspace && git clone https://github.com/HemishJain09/PI-CAI-Baseline.git baseline
else:
    !cd /tmp/PI-CAI_Workspace/baseline && git pull

if not os.path.exists("/tmp/PI-CAI_Workspace/Z-SSMNet"):
    !cd /tmp/PI-CAI_Workspace && git clone https://github.com/yuanyuan29/Z-SSMNet.git Z-SSMNet

print("Repositories ready.")

In [ ]:
# 4. Install dependencies
import os
os.environ["SKLEARN_ALLOW_DEPRECATED_SKLEARN_PACKAGE_INSTALL"] = "True"
!pip install -q -r $WORKSPACE_DIR/requirements.txt
!pip install -q git+https://github.com/DIAGNijmegen/nnUNet.git@1.7.0-3
print("Dependencies installed.")

In [ ]:
# 5. Run the full Domain Adaptation Pipeline
import os
os.chdir(os.environ["WORKSPACE_DIR"])
os.environ["SKLEARN_ALLOW_DEPRECATED_SKLEARN_PACKAGE_INSTALL"] = "True"

# Execute pipeline
!chmod +x run_nnunet_pipeline.sh
!./run_nnunet_pipeline.sh